In [1]:
pip install confluent-kafka

     |████████████████████████████████| 2.7 MB 31.2 MB/s 


In [2]:
!curl --output kfk.tgz https://downloads.apache.org/kafka/3.0.0/kafka_2.13-3.0.0.tgz
!tar -xvf ./kfk.tgz
!./kafka_2.13-3.0.0/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.0.0/config/zookeeper.properties
!./kafka_2.13-3.0.0/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.0.0/config/server.properties
!ps -ef | grep kafka

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 82.3M  100 82.3M    0     0  20.2M      0  0:00:04  0:00:04 --:--:-- 20.2M
kafka_2.13-3.0.0/
kafka_2.13-3.0.0/LICENSE
kafka_2.13-3.0.0/NOTICE
kafka_2.13-3.0.0/bin/
kafka_2.13-3.0.0/bin/kafka-delete-records.sh
kafka_2.13-3.0.0/bin/trogdor.sh
kafka_2.13-3.0.0/bin/connect-mirror-maker.sh
kafka_2.13-3.0.0/bin/kafka-console-consumer.sh
kafka_2.13-3.0.0/bin/kafka-consumer-perf-test.sh
kafka_2.13-3.0.0/bin/kafka-log-dirs.sh
kafka_2.13-3.0.0/bin/zookeeper-server-stop.sh
kafka_2.13-3.0.0/bin/kafka-verifiable-consumer.sh
kafka_2.13-3.0.0/bin/kafka-features.sh
kafka_2.13-3.0.0/bin/kafka-acls.sh
kafka_2.13-3.0.0/bin/zookeeper-server-start.sh
kafka_2.13-3.0.0/bin/kafka-server-stop.sh
kafka_2.13-3.0.0/bin/kafka-configs.sh
kafka_2.13-3.0.0/bin/kafka-reassign-partitions.sh
kafka_2.13-3.0.0/bin/kafka-leader-election.sh
kafka_2.13-3.0.0/bin/kaf

In [12]:
import os
from datetime import datetime
import time
import threading
from confluent_kafka import Producer, Consumer
import socket
import string, sys
from collections import Counter
from string import punctuation
import nltk
import json
from multiprocessing import Pool, Process, Manager, Lock
from functools import partial
nltk.download('punkt')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
swords = stopwords.words("english")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [98]:
NO_PARTITIONS = 3
NO_CONSUMERS = 2

In [87]:
from confluent_kafka.admin import AdminClient, NewTopic


admin_client = AdminClient({
    "bootstrap.servers": "localhost:9092"
})

topic_list = []
topic_list.append(NewTopic("TOPIC", NO_PARTITIONS, 1))
admin_client.create_topics(topic_list)
# admin_client.delete_topics(["TOPIC"])

{'TOPIC': <Future at 0x7f18927e6bd0 state=running>}

In [88]:
conf = {'bootstrap.servers': "localhost:9092",
        'client.id': socket.gethostname()}

producer = Producer(conf)
def delivery_report(err, msg):
  """ Called once for each message produced to indicate delivery result.
      Triggered by poll() or flush(). """
  if err is not None:
      print("Failed to deliver message: %s: %s" % (str(msg), str(err)))

In [89]:
filename="/content/drive/MyDrive/Colab Data/data.txt"
text_file = open(filename, "r")
lines= text_file.readlines()
text_file.close()
nrows=len(lines)

In [105]:
start = time.time()
for line in lines:
  twords = word_tokenize(line)
  for word in twords:
    producer.poll(0)
    producer.produce("TOPIC", word.encode('utf-8'), callback=delivery_report)
producer.flush()
end1 = time.time()
print("\ntime spent for producer: ", end1-start)


time spent for producer:  5.9872963428497314


In [106]:
def create_consumer_pool(count, iterable):
  lock = Lock()
  consumer = Consumer({
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'mygroup',
    'auto.offset.reset': 'earliest'
  })
  running = True
  start = time.time()
  try:
    consumer.subscribe(['TOPIC'])
    while running:
      msg = consumer.poll(1.0)
      if msg is None:
        running = False
        continue
      if msg.error():
        print("Consumer error: {}".format(msg.error()))
        continue
      else:
        try:
          lock.acquire()
          output = msg.value().decode()
          if output in count.keys():
            count[output] = count[output] + 1
          else:
            count[output] = 1
        finally:
          lock.release()

  finally:
    consumer.close()

In [107]:
manager = Manager()
count = manager.dict()
start = time.time()
with Pool(NO_CONSUMERS) as pool:
    word_tokens = pool.map(partial(create_consumer_pool, count), range(NO_CONSUMERS))

end1 = time.time()
print("\ntime spent for consumers: ", float(end1-start)-1.0)
print("count: ", count)


time spent for consumers:  3.1438939571380615
count:  {'it': 96, 'made': 12, 'a': 159, 'sickening': 1, 'impression': 3, '.': 235, 'She': 17, 'seemed': 3, 'to': 208, 'Raskolnikov': 10, 'about': 18, 'thirty': 3, 'years': 6, 'old': 6, 'and': 259, 'was': 79, 'certainly': 1, 'strange': 1, 'wife': 4, 'for': 73, 'Marmeladov': 5, '...': 9, 'had': 53, 'not': 53, 'heard': 7, 'them': 12, 'did': 10, 'notice': 2, 'coming': 3, 'in': 120, 'be': 52, 'lost': 3, 'thought': 9, ',': 491, 'hearing': 1, 'seeing': 6, 'nothing': 10, 'The': 19, 'room': 15, 'close': 4, 'but': 43, 'she': 64, 'opened': 3, 'the': 215, 'window': 2, ';': 37, 'stench': 1, 'rose': 1, 'from': 24, 'staircase': 1, 'door': 5, 'on': 44, 'stairs': 2, 'closed': 1, 'From': 3, 'inner': 2, 'rooms': 1, 'clouds': 1, 'of': 142, 'tobacco': 1, 'smoke': 1, 'floated': 1, 'kept': 3, 'coughing': 1, 'youngest': 1, 'child': 2, 'girl': 8, 'six': 4, 'asleep': 4, 'sitting': 2, 'curled': 1, 'up': 26, 'floor': 3, 'with': 52, 'her': 63, 'head': 5, 'sofa': 5, '